# 01 - Bench capture overview

**Question:** what do the first O4/ELRS bench captures look like — and are
they clean enough to publish features from?

**Scope (receive-only):** HackRF recordings of bench transmitters
(RadioMaster Boxer on ELRS 2.4 GHz, DJI O4 video link). Everything here is
analysis of received data: nothing in this notebook transmits, jams or
otherwise interferes.

**Controls:** every session should include a TX-OFF capture
(`protocol: "noise"` in the manifest). Signal features are only trusted in
comparison against that control, never standalone.

**SYNTHETIC fallback:** with no entries under
`purpose = signature_capture`, an in-memory synthetic capture
(band-limited noise bursts + DC offset + a few clipped samples) exercises
the whole pipeline. All synthetic output is labeled SYNTHETIC — it is NOT
measurements.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal

from tacet.dsp import features
from tacet.loaders import cs8
from tacet.loaders import manifest as manifest_mod

# --- Configuration --------------------------------------------------------
SEGMENT_S = 2.0           # features are computed on the first N seconds
NFFT = 1024
THRESHOLD_REL_DB = -35.0  # duty/burst threshold rel. to the envelope peak
QC_BLOCK = 10_000_000     # complex samples per streamed QC block
SYNTHETIC_FALLBACK = True # demo pipeline when the manifest has no captures
print("config ok")


In [ ]:
# --- Load: manifest entries, or SYNTHETIC in-memory fallback ---------------
captures = []

rows = manifest_mod.query(purpose="signature_capture") if Path("manifest.json").exists() else []
for e in rows:
    for ch in e["channels"]:
        captures.append({"id": e["id"], "path": ch["path"],
                         "fs": e["sample_rate_sps"],
                         "protocol": e["protocol"],
                         "notes": e["operator_notes"],
                         "synthetic": False})

if not captures and SYNTHETIC_FALLBACK:
    # SYNTHETIC fallback — NOT measurements. In memory only, never written.
    # Protocol stays neutral ("other"): band-limited noise is not OFDM.
    fs = 1_000_000.0
    dur_s = 2.0
    n = int(fs * dur_s)
    t = np.arange(n) / fs
    rng = np.random.default_rng(11)
    base = rng.standard_normal(n) + 1j * rng.standard_normal(n)
    taps = signal.firwin(255, 100e3, fs=fs)          # real lowpass, 100 kHz
    tt = np.arange(taps.size) / fs
    bc = taps * np.exp(2j * np.pi * 150e3 * tt)      # analytic band 100-200 kHz
    band = signal.lfilter(bc, 1.0, base)
    burst_gate = (np.mod(t, 0.5) < 0.15).astype(float)   # 30% duty bursts
    z = band * burst_gate + 50.0                     # + DC offset (HackRF spike)
    clip_idx = rng.choice(n, size=20, replace=False)
    z[clip_idx] = 300.0 + 1j * 300.0                 # a few clipped samples
    z = z.astype(np.complex64)
    captures.append({"id": "synthetic_bench", "path": None, "z": z, "fs": fs,
                     "protocol": "other", "notes": "SYNTHETIC in-memory demo",
                     "synthetic": True})
    print("SYNTHETIC fallback — NOT measurements")
print(f"captures loaded: {[c['id'] for c in captures]}")


In [ ]:
# --- QC, features and plots (one pass; heavy arrays are not retained) ------
# rail_fraction is streamed over the WHOLE capture (int8 rails = clipping);
# spectrogram / PSD / features run on the first SEGMENT_S seconds. The dB
# spectrogram is derived from the linear S computed for the features: the
# spectrogram is computed exactly once per capture and released after the
# plot, so a long session stays flat in memory.
PROTO_LABEL = {"dji_o4": "DJI O4 (OFDM)", "dji_o3": "DJI O3 (OFDM)",
               "elrs": "ELRS (CSS chirps)", "analog_fpv": "analog FPV",
               "noise": "TX-OFF control"}
qc_rows = []
controls = [c for c in captures if c["protocol"] == "noise"]
for cap in captures:
    if cap["path"] is not None:
        n_rail, n_total = 0.0, 0
        for block in cs8.iter_cs8_blocks(cap["path"], QC_BLOCK):
            n_rail += features.rail_fraction(block) * block.size
            n_total += block.size
        cap["rail"] = n_rail / n_total if n_total else 0.0
        z = cs8.load_cs8(cap["path"], n_samples=int(cap["fs"] * SEGMENT_S))
    else:
        cap["rail"] = features.rail_fraction(cap["z"])
        z = cap["z"][: int(cap["fs"] * SEGMENT_S)]

    y = features.remove_dc(z)
    del z
    f_w, p = features.welch_psd(y, cap["fs"], nfft=NFFT)
    cap["p"] = p   # light (NFFT floats): kept for the TX-OFF comparison
    bw, f_lo, f_hi = features.occupied_bandwidth(f_w, p, percent=99.0)
    f_s, t_s, S = features.spectrogram(y, cap["fs"], nfft=NFFT)
    env = features.time_envelope(S)
    duty = features.duty_cycle(env, threshold_rel_db=THRESHOLD_REL_DB)
    bursts = features.burst_structure(env, t_s, threshold_rel_db=THRESHOLD_REL_DB)

    qc_rows.append({
        "capture": cap["id"],
        "protocol": cap["protocol"],
        "synthetic": cap["synthetic"],
        "rail_fraction_whole": cap["rail"],
        "flatness": features.spectral_flatness(p),
        "obw_hz": bw,
        "obw_lo_hz": f_lo,
        "obw_hi_hz": f_hi,
        "duty": duty,
        "n_bursts": len(bursts["bursts"]),
        "mean_burst_s": bursts["mean_burst_s"],
        "mean_gap_s": bursts["mean_gap_s"],
    })
    flag = "RAIL QC FAIL (> 0.1%) - invalid capture" if cap["rail"] > 1e-3 else "rail qc ok"
    print(f"{cap['id']}: rail_fraction(whole) = {cap['rail']:.6%}  [{flag}]")

    label = cap["id"] + (" (SYNTHETIC)" if cap["synthetic"] else "")
    proto = ("synthetic bench signal" if cap["synthetic"]
             else PROTO_LABEL.get(cap["protocol"], cap["protocol"]))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    im = ax1.pcolormesh(t_s, f_s / 1e6, 10.0 * np.log10(S + 1e-30),
                        shading="auto", cmap="viridis")
    fig.colorbar(im, ax=ax1, label="dB")
    ax1.set_xlabel("t [s]")
    ax1.set_ylabel("f [MHz]")
    ax1.set_title(f"{proto} — dB spectrogram ({label})")
    ax2.plot(f_w / 1e6, 10.0 * np.log10(p + 1e-30), lw=0.6)
    ax2.set_xlabel("f [MHz]")
    ax2.set_ylabel("PSD [dB]")
    ax2.set_title(f"{proto} — Welch PSD ({label})")
    fig.tight_layout()
    plt.show()
    del y, S, env   # heavy arrays released once the plot is produced

    if cap["protocol"] == "elrs":
        print(f"ELRS window-coverage note ({cap['id']}): the hop sequence "
              "visits the whole 2.4 GHz band, so a single 20 MHz window only "
              "sees the hops that land inside it — duty/burst numbers here "
              "describe THIS window, not the whole link.")

# TX-OFF control comparison when the session has one (protocol == "noise").
signals = [c for c in captures if c["protocol"] != "noise"]
if controls and signals:
    p_ctrl = float(np.mean(controls[0]["p"]))
    for c in signals:
        snr_db = 10.0 * np.log10(float(np.mean(c["p"])) / p_ctrl)
        print(f"TX-OFF control comparison: {c['id']} vs {controls[0]['id']} "
              f"-> mean PSD ratio {snr_db:+.1f} dB")
else:
    print("no TX-OFF (protocol='noise') control capture in this session "
          "- comparison skipped")

df = pd.DataFrame(qc_rows)
df


## Interpretation guide

- **OFDM-like video (DJI O4/O3):** spectral flatness near 1 with a wide 99%
  occupied bandwidth and a bursty duty envelope (video goes out in frames).
  A pure tone would show flatness near 0 with all power in a handful of
  bins; white noise shows flatness near 1 too, which is why the TX-OFF
  control comparison decides whether the wide band is signal.
- **ELRS control link:** LoRa/CSS chirps show up as diagonal ridges in the
  spectrogram, one sweep per hop; in the Welch PSD each hop widens into a
  modest hump, easily mistaken for noise. The ridge structure in the
  time-frequency view is the discriminator, and the hopping means a single
  center-frequency window under-reports the link's duty cycle.
- **Rail QC:** `rail_fraction` over the whole capture counts samples pinned
  at the int8 rails. Above 0.1% (0.001) the capture is clipping hard:
  the capture is invalid — drop LNA/VGA (8 dB / 2 dB steps) and re-record
  instead of publishing features from it.
- **DC spike:** HackRF captures carry a center spike; `remove_dc` plus the
  DC-bin exclusion inside `time_envelope`/`spectral_flatness` keep it out
  of every power statistic above.
